# Chapter 4 Source Code Notebook

This notebook builds the source code for Chapter 4, **From Pipelines to Agents**.

The code builds a controlled observe, decide, act agent loop with actions, policies, tools, and fallback behavior for SupportOps AI.

Before running the notebook, install any required packages and set the required API keys in your environment where model powered examples are used.


## Step 1: Typed Tool Contracts

### `agent/tools/contracts.py`

The agent starts with typed tool contracts so every available action has a clear input and output shape. This keeps tool use bounded before the agent is allowed to choose actions.

In [ ]:
# agent/tools/contracts.py
from dataclasses import dataclass, field
from enum import Enum
from typing import Any, Optional

class ActionRisk(Enum):
    READ        = "read"         # free - no side effects
    WRITE       = "write"        # logged - modifies internal state
    SIDE_EFFECT = "side_effect"  # gated - external consequence

@dataclass
class ToolResult:
    """Standard feedback envelope returned by every tool."""
    success:    bool
    data:       Any
    error:      Optional[str] = None
    source:     str = ""         # which tool/system produced this
    latency_ms: float = 0.0

@dataclass
class ToolSpec:
    """The contract for a single tool."""
    name:        str
    description: str
    risk:        ActionRisk
    input_keys:  list[str]    # required keys the agent must provide
    output_desc: str          # human-readable description of output
    max_retries: int = 2
    requires_confidence: str = "low"  # minimum confidence to call


## Step 2: The Tool Registry

### `agent/tools/registry.py`

After the contracts are defined, the registry gives the agent a controlled list of tools it can call. The agent can discover tools through this interface without gaining unrestricted access to everything in the system.

In [ ]:
# agent/tools/registry.py
import time
from typing import Callable
from .contracts import ToolSpec, ToolResult, ActionRisk

class ToolRegistry:
    def __init__(self):
        self._tools: dict[str, tuple[ToolSpec, Callable]] = {}

    def register(self, spec: ToolSpec,
                 fn: Callable[[dict], ToolResult]) -> None:
        self._tools[spec.name] = (spec, fn)

    def spec(self, name: str) -> ToolSpec:
        if name not in self._tools:
            raise KeyError(f"Unknown tool: {name}")
        return self._tools[name][0]

    def all_specs(self) -> list[ToolSpec]:
        return [spec for spec, _ in self._tools.values()]

    def execute(self, name: str,
                inputs: dict) -> ToolResult:
        if name not in self._tools:
            return ToolResult(
                success=False, data=None,
                error=f"Tool not found: {name}"
            )
        spec, fn = self._tools[name]
        # Validate required inputs
        missing = [k for k in spec.input_keys if k not in inputs]
        if missing:
            return ToolResult(
                success=False, data=None,
                error=f"Missing required inputs: {missing}"
            )
        start = time.time()
        result = fn(inputs)
        result.latency_ms = (time.time() - start) * 1000
        return result


## Step 3: The Action Policy

### `agent/policy.py`

The policy layer then decides which actions are allowed under different risk conditions. This ensures the agent can choose actions only within boundaries that the system enforces.

In [ ]:
# agent/policy.py
from dataclasses import dataclass
from .tools.contracts import ActionRisk, ToolSpec
from ..session import Session

@dataclass
class PolicyDecision:
    approved:  bool
    reason:    str
    risk_tier: str

class ActionPolicy:
    def __init__(self,
                 allowed_risk: ActionRisk = ActionRisk.SIDE_EFFECT,
                 require_confidence_for_side_effects: str = "high"):
        self.allowed_risk = allowed_risk
        self.confidence_gate = require_confidence_for_side_effects
        # track calls this session to prevent duplication
        self._call_log: list[tuple[str, str]] = []

    def check(self, tool_name: str, inputs: dict,
               spec: ToolSpec, session: Session,
               current_confidence: str = "high") -> PolicyDecision:

        # 1. Check risk tier authorization
        risk_levels = {
            ActionRisk.READ: 0,
            ActionRisk.WRITE: 1,
            ActionRisk.SIDE_EFFECT: 2
        }
        if risk_levels[spec.risk] > risk_levels[self.allowed_risk]:
            return PolicyDecision(
                False,
                f"{tool_name} exceeds allowed risk tier",
                spec.risk.value
            )

        # 2. Confidence gate for side-effect actions
        confidence_rank = {"low": 0, "medium": 1, "high": 2}
        if spec.risk == ActionRisk.SIDE_EFFECT:
            if confidence_rank.get(current_confidence, 0) < \
               confidence_rank.get(self.confidence_gate, 2):
                return PolicyDecision(
                    False,
                    f"Side-effect action requires {self.confidence_gate} "+
                    f"confidence, got {current_confidence}",
                    spec.risk.value
                )

        # 3. Deduplication check (exact same call this session)
        call_sig = f"{tool_name}:{sorted(inputs.items())}"
        if call_sig in [sig for _, sig in self._call_log]:
            return PolicyDecision(
                False,
                f"{tool_name} already called with identical inputs",
                spec.risk.value
            )
        self._call_log.append((tool_name, call_sig))
        return PolicyDecision(True, "approved", spec.risk.value)


## Step 4: The Agent Controller

### `agent/controller.py`

With tools and policy ready, the controller runs the observe, decide, act loop. It reads the state, selects an action, executes it, handles failures, and records each decision.

In [ ]:
# agent/controller.py
import json
import time
from dataclasses import dataclass
from typing import Optional
import anthropic
from .tools.registry import ToolRegistry
from .tools.contracts import ActionRisk
from .policy import ActionPolicy
from ..session import Session, SessionStatus
from ..reasoning.budget import BudgetEnforcer
from ..reasoning.logger import StepLogger
from ..tracker import SystemTrace
from ..config import SystemConfig, DEFAULT_CONFIG

DECIDE_PROMPT_TEMPLATE = """
You are a support resolution agent. Your goal: {goal}

Current session state:
{session_summary}

Available actions:
{actions_list}

Last action result:
{last_result}

Choose the SINGLE best next action. Return JSON:
{
  "action": "<tool_name or COMPLETE or ESCALATE>",
  "inputs": {{...}},
  "confidence": "high|medium|low",
  "reasoning": "<one sentence>"
}
Return ONLY valid JSON."""

@dataclass
class AgentDecision:
    action:     str
    inputs:     dict
    confidence: str
    reasoning:  str

class AgentController:
    def __init__(
        self,
        client:   anthropic.Anthropic,
        registry: ToolRegistry,
        policy:   ActionPolicy,
        goal:     str,
        config:   SystemConfig = DEFAULT_CONFIG
    ):
        self.client   = client
        self.registry = registry
        self.policy   = policy
        self.goal     = goal
        self.config   = config

    def _observe(self, session: Session,
                 last_result: Optional[dict]) -> str:
        """Assemble the observation context for the decide step."""
        actions_list = "\n".join([
            f"  - {s.name}: {s.description} [risk: {s.risk.value}]"
            for s in self.registry.all_specs()
        ])
        session_summary = (
            f"Status: {session.status.value}\n"
            f"Routing: {session.routing_category}\n"
            f"Classification: {
                session.classification.category.value
                if session.classification else 'not yet classified'
            }\n"
            f"Steps taken: {len(session.actions_taken)}\n"
            f"Escalated: {session.escalated}"
        )
        return DECIDE_PROMPT_TEMPLATE.format(
            goal=self.goal,
            session_summary=session_summary,
            actions_list=actions_list,
            last_result=json.dumps(last_result or {})
        )

    def _decide(self, observation: str,
                trace: SystemTrace) -> AgentDecision:
        """Ask the model to choose the next action."""
        from ..tracker import tracked_call
        response = tracked_call(
            self.client, "agent_decide", trace,
            model=self.config.smart_model,
            max_tokens=512,
            messages=[{"role": "user", "content": observation}]
        )
        data = json.loads(response.content[0].text.strip())
        return AgentDecision(
            action     = data["action"],
            inputs     = data.get("inputs", {}),
            confidence = data.get("confidence", "medium"),
            reasoning  = data.get("reasoning", "")
        )

    def run(self, session: Session,
             trace: SystemTrace,
             budget: BudgetEnforcer,
             logger: StepLogger) -> Session:
        last_result = None
        session.status = SessionStatus.PROCESSING

        while True:
            # ── Budget check (always first) ──────────────────────────
            bstatus = budget.tick()
            if bstatus.exceeded:
                logger.log("budget_exceeded", {},
                    {"steps": bstatus.step_count,
                     "elapsed": bstatus.elapsed_seconds},
                    "failed", notes="Hard stop by budget enforcer")
                session.status = SessionStatus.FAILED
                session.escalated = True
                session.escalation_reason = "budget_exceeded"
                break

            # ── Observe ──────────────────────────────────────────────
            observation = self._observe(session, last_result)

            # ── Decide ───────────────────────────────────────────────
            t0 = time.time()
            decision = self._decide(observation, trace)
            decide_ms = (time.time() - t0) * 1000

            logger.log(
                "agent_decide",
                inputs  = {"step": bstatus.step_count},
                outputs = {"action": decision.action,
                           "confidence": decision.confidence,
                           "reasoning": decision.reasoning},
                status  = "success",
                confidence = decision.confidence,
                latency_ms = decide_ms
            )

            # ── Check model-declared terminal actions ────────────────
            if decision.action == "COMPLETE":
                # Validate completion before accepting
                if session.draft_response:
                    session.status = SessionStatus.RESOLVED
                    logger.log("complete", {}, {"status": "resolved"},
                               "success")
                    break
                else:
                    # Model said complete but no draft exists
                    # Feed back as an observation, don't stop
                    last_result = {"error":
                        "COMPLETE declared but draft_response is empty."}
                    continue

            if decision.action == "ESCALATE":
                session.status = SessionStatus.ESCALATED
                session.escalated = True
                session.escalation_reason = (
                    f"agent_requested: {decision.reasoning}"
                )
                logger.log("escalate", {},
                    {"reason": session.escalation_reason}, "success")
                break

            # ── Policy check ─────────────────────────────────────────
            try:
                spec = self.registry.spec(decision.action)
            except KeyError:
                last_result = {"error":
                    f"Unknown action: {decision.action}"}
                continue

            policy_result = self.policy.check(
                decision.action, decision.inputs,
                spec, session, decision.confidence
            )
            if not policy_result.approved:
                logger.log(
                    "policy_block",
                    inputs  = {"action": decision.action},
                    outputs = {"reason": policy_result.reason},
                    status  = "failed",
                    notes   = "Action blocked by policy"
                )
                last_result = {
                    "blocked": True,
                    "reason": policy_result.reason
                }
                continue

            # ── Act ──────────────────────────────────────────────────
            t0 = time.time()
            tool_result = self.registry.execute(
                decision.action, decision.inputs
            )
            act_ms = (time.time() - t0) * 1000

            logger.log(
                f"act:{decision.action}",
                inputs  = decision.inputs,
                outputs = {"success": tool_result.success,
                           "data_preview": str(tool_result.data)[:100],
                           "error": tool_result.error},
                status  = "success" if tool_result.success else "failed",
                latency_ms = act_ms
            )

            # Reflect tool result back as next observation
            last_result = {
                "tool": decision.action,
                "success": tool_result.success,
                "data": str(tool_result.data)[:500],
                "error": tool_result.error
            }

            # Write tool outputs to session state where applicable
            if decision.action == "classify_ticket" and tool_result.success:
                session.classification = tool_result.data
            elif decision.action == "draft_response" and tool_result.success:
                session.draft_response = tool_result.data

        return session


## Step 5: Registering Tools and Running the Agent

### `agent/run_agent.py`

The final cell registers the available tools and runs the controlled agent loop. This shows how the contracts, registry, policy, and controller work together in a complete flow.

In [ ]:
# agent/run_agent.py
import os
import anthropic
from supportops_ai.agent.tools.registry import ToolRegistry
from supportops_ai.agent.tools.contracts import (
    ToolSpec, ToolResult, ActionRisk
)
from supportops_ai.agent.policy import ActionPolicy
from supportops_ai.agent.controller import AgentController
from supportops_ai.session import Session
from supportops_ai.memory.in_memory import InMemoryStore
from supportops_ai.reasoning.budget import BudgetEnforcer
from supportops_ai.reasoning.logger import StepLogger
from supportops_ai.tracker import SystemTrace
from supportops_ai.config import DEFAULT_CONFIG
from supportops_ai.services.classifier import classify_ticket
from supportops_ai.services.drafter import draft_response
from supportops_ai.services.escalation import apply_escalation_rules

client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
config = DEFAULT_CONFIG

# ── Register tools ───────────────────────────────────────────────────
registry = ToolRegistry()

registry.register(
    ToolSpec(
        name="classify_ticket",
        description="Classify ticket intent, urgency, and sentiment",
        risk=ActionRisk.READ,
        input_keys=["ticket_text"],
        output_desc="TicketClassification object"
    ),
    lambda inputs: ToolResult(
        success=True,
        data=classify_ticket(
            client, inputs["ticket_text"],
            SystemTrace("tool"), config
        ),
        source="classifier"
    )
)

registry.register(
    ToolSpec(
        name="check_escalation",
        description="Apply escalation rules to current session",
        risk=ActionRisk.READ,
        input_keys=[],
        output_desc="Updated session with escalation status set"
    ),
    lambda inputs: ToolResult(success=True, data="checked",
                              source="escalation_rules")
)

registry.register(
    ToolSpec(
        name="draft_response",
        description="Draft a customer response for the current ticket",
        risk=ActionRisk.WRITE,
        input_keys=[],
        output_desc="Draft response string",
        requires_confidence="medium"
    ),
    lambda inputs: ToolResult(
        success=True,
        data="[Draft would be generated here]",
        source="drafter"
    )
)

registry.register(
    ToolSpec(
        name="send_response",
        description="Send the drafted response to the customer",
        risk=ActionRisk.SIDE_EFFECT,
        input_keys=["draft"],
        output_desc="Send confirmation",
        requires_confidence="high"
    ),
    lambda inputs: ToolResult(
        success=True,
        data={"sent": True, "preview": inputs["draft"][:80]},
        source="email_service"
    )
)

# ── Build agent ──────────────────────────────────────────────────────
policy = ActionPolicy(
    allowed_risk=ActionRisk.WRITE,          # no side effects without approval
    require_confidence_for_side_effects="high"
)

controller = AgentController(
    client=client,
    registry=registry,
    policy=policy,
    goal="Classify the ticket, check escalation rules, and draft a response.",
    config=config
)

# ── Run ─────────────────────────────────────────────────────────────
if __name__ == "__main__":
    ticket = (
        "Our entire team lost access to the dashboard after your "
        "maintenance window yesterday. We have a demo in two hours "
        "and we cannot log in. This is critical."
    )

    trace   = SystemTrace("TKT-004")
    session = Session(ticket_id="TKT-004", raw_input=ticket)
    budget  = BudgetEnforcer(max_seconds=120.0, max_steps=10)
    logger  = StepLogger(session)

    session = controller.run(session, trace, budget, logger)

    print(logger.trace())
    print(f"\nFinal status: {session.status.value}")
    policy_blocks = len([l for l in logger._logs if l.step_name == "policy_block"])
    print(f"Policy blocks: {policy_blocks}")
    if session.draft_response:
        print(f"\nDraft:\n{session.draft_response}")
